# GameSense baseline recommender smoke run (Kaggle-compatible)

This notebook runs the current local baseline recommender and offline evaluation on the committed self-authored sample fixture by default. It is intended for project-flow validation only: it does not download Kaggle datasets, upload credentials, train/fine-tune a model, generate embeddings, or claim real-catalog accuracy.

If you attach data in Kaggle later, first add a source manifest and pass the existing source-check/license/terms validation. The current gate accepts only the approved self-authored sample.

In [ ]:
from pathlib import Path
import os

# In Kaggle, upload or clone the GameSense repository and set GAMESENSE_REPO_ROOT if needed.
REPO_ROOT = Path(os.getenv('GAMESENSE_REPO_ROOT', '.')).resolve()
assert (REPO_ROOT / 'backend/app/recommender.py').exists(), f'GameSense repo not found at {REPO_ROOT}'
os.chdir(REPO_ROOT)
print('repo:', REPO_ROOT)

In [ ]:
import sys, subprocess

# Install only the repository requirements; no external datasets or credentials are used.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'backend/requirements.txt'])
sys.path.insert(0, str(REPO_ROOT / 'backend'))

In [ ]:
from pathlib import Path
import json
from app.ml_pipeline import source_check

MANIFEST_PATH = Path(os.getenv('GAMESENSE_SOURCE_MANIFEST', 'data/source-manifests/example.self-authored.json'))
source_report = source_check(MANIFEST_PATH, REPO_ROOT)
print(json.dumps(source_report, indent=2, sort_keys=True))
assert source_report['ok'], 'Source/license gate failed; do not run evaluation on this data.'

In [ ]:
from app.ml_evaluation import evaluate_baseline

evaluation = evaluate_baseline()
print(json.dumps({
    'ok': evaluation['ok'],
    'scope': evaluation['scope'],
    'metrics': evaluation['metrics'],
    'limitations': evaluation['limitations'],
}, indent=2, sort_keys=True))
assert evaluation['scope']['externalDataUsed'] is False
assert evaluation['scope']['realCatalogEvidence'] is False

In [ ]:
OUTPUT_DIR = Path(os.getenv('GAMESENSE_OUTPUT_DIR', '/kaggle/working' if Path('/kaggle/working').exists() else '/tmp'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'gamesense-baseline-evaluation.json'
output_path.write_text(json.dumps(evaluation, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('wrote', output_path)